In [25]:
# Build main diff-in-diff analysis data

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "tax_analysis_panel.parquet")


In [26]:
# Pre-process ALFIN data

alfin_xwalk = pd.read_csv(os.path.join(MY_DATA_PATH, "raw_data", "alfin_crosswalk.csv"))
alfin_xwalk = alfin_xwalk.loc[alfin_xwalk['UNIT_TYPE_CODE']==2].reset_index(drop=True)  # keep only the cities
alfin_df = pd.read_parquet(os.path.join(MY_DATA_PATH, "raw_data", "alfin_raw.parquet"))
alfin_df['ID'] = alfin_df['ID'].astype('int64')

# Keep following item codes:
# T19: Other Selective Sales Tax (Lodging taxes go here)
# T28: Occupational and Business Licenses, NEC (STR licenses may go here)
# T29: Other License Taxes (STR licenses may go here)

alfin_df = alfin_df.loc[ alfin_df['ITEM_CODE'].isin(['T19', 'T28', 'T29'])].reset_index(drop=True)
alfin_df = alfin_xwalk[['city', 'state', 'ID']].merge(
    alfin_df[['ID', 'YEAR', 'ITEM_CODE', 'AMOUNT']], 
    on='ID', how='left'
)

alfin_wide = alfin_df.pivot(
    index=['city', 'state', 'ID', 'YEAR'], columns='ITEM_CODE', values='AMOUNT'
).sort_values(
    by=['city', 'state', 'YEAR']
).reset_index()

for col in ['T19', 'T28', 'T29']:
    alfin_wide[col] = alfin_wide[col].fillna(0)



In [27]:
# Load other data

regs_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/best_treatment_dates_2026-07.csv"))
tax_df = pd.read_excel(os.path.join(RAW_DATA_PATH, "lincoln-institute/FiSC-Full-Dataset-2023-Update.xlsx"), sheet_name="Data")
zhvi_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "zhvi/City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))

mapping = pd.read_csv(os.path.join(MY_DATA_PATH, "city_mapping.csv"))

In [28]:
# Reshape zhvi data long by year

id_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
           'StateName', 'State', 'Metro', 'CountyName']

zhvi_long = zhvi_df.melt(
    id_vars=id_cols,
    var_name = 'date',
    value_name = 'ZHVI'
)

zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])
zhvi_long['year'] = zhvi_long['date'].dt.year

zhvi_long = zhvi_long.groupby(id_cols + ['year']).agg({'ZHVI': 'mean'}).reset_index()


In [29]:
# Merging the data

df = regs_df.merge(
    mapping.rename(columns={'policy_city': 'city'}),
    on='city',
    how='left'
)

df = df.merge(
    tax_df.rename(columns={'city_name': 'tax_city'}),
    on='tax_city',
    how='left'
)

df = df.merge(
    zhvi_long,
    on=['RegionID', 'year'],
    how='inner'
)

df = df.merge(
    alfin_wide[['city', 'state', 'YEAR', 'T19', 'T28', 'T29']].rename(columns={'YEAR': 'year'}),
    how='left'
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Columns: 567 entries, city to T29
dtypes: float64(546), int64(5), str(16)
memory usage: 5.3 MB


In [30]:
# clean dates

df['best_enforcement'] = pd.to_datetime(df['best_enforcement'], errors='coerce')
df['best_passage'] = pd.to_datetime(df['best_passage'], errors='coerce')

df['enforcement_year'] = df['best_enforcement'].dt.year
df['passage_year'] = df['best_passage'].dt.year

df['years_from_enforcement'] = (df['year'] - df['enforcement_year'])
df['years_from_passage'] = (df['year'] - df['passage_year'])


In [31]:
# sample selection

mask = (np.abs(df['years_from_enforcement']) > 6) & (df['years_from_enforcement'].notna())
df = df.loc[~mask].reset_index(drop=True)

# for cities without an enforcement date, drop years outside the min/max of the remaining data

year_min = df.loc[df['best_enforcement'].notna(), 'year'].min()
year_max = df.loc[df['best_enforcement'].notna(), 'year'].max()
mask = (df['year'] < year_min) | (df['year'] > year_max)
df = df.loc[~mask].reset_index(drop=True)

In [32]:
# change enforcement and passage year to 0 for cities without enforcement/passage dates
# (standard convention for CSDID package in R)

df.loc[df['best_enforcement'].isna(), 'enforcement_year'] = 0
df.loc[df['best_passage'].isna(), 'passage_year'] = 0

df['enforcement_year'] = df['enforcement_year'].astype(int)
df['passage_year'] = df['passage_year'].astype(int)


In [33]:
# make a city_id integer (also required for CSDID package)

df['city_id'] = df['city'].astype('category').cat.codes

In [34]:
# output dataframe for analysis

df.to_parquet(OUTPUT_FILEPATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 572 entries, 0 to 571
Columns: 572 entries, city to city_id
dtypes: datetime64[us](2), float64(548), int64(7), int8(1), str(14)
memory usage: 2.6 MB
